# Highway Networks — Highway Networks (2015)

_Papers · Computer Vision_

**A learned sigmoid gate lets a layer choose, per-unit, between transforming its input and carrying it through unchanged — so very deep nets finally train.**

---

This notebook builds a highway network from scratch, one step at a time — the gated highway block, a matched plain block, a 40-layer net of each, and a training run that shows the gate (with its negative carry-bias) is what lets depth train. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build the highway network in four steps: (0) check the lesson's worked gate example, (1) the highway block and a matched plain block, (2) a 40-layer net of either kind, (3) a tiny blob dataset plus a training run comparing plain, highway, and a no-carry-bias ablation.

### Step 0 — Imports and the worked gate example

A highway unit blends two paths with a sigmoid gate `T`: `y = H*T + x*(1-T)`. When `T` is near 1 the transform `H` dominates; near 0 the raw input `x` is carried through. Here we reproduce the lesson's element-wise worked example to see both behaviours at once.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked example: y = H*T + x*(1-T), T = sigmoid(z). ---
x = torch.tensor([1.0, -2.0])
Hx = torch.tensor([0.8, 0.5])              # the transform's ReLU output
z = torch.tensor([2.0, -1.5])             # the gate's pre-activation W_T^T x + b_T
T = torch.sigmoid(z)                       # gate in (0,1):  ~[0.8808, 0.1824]
y = Hx * T + x * (1 - T)                   # Eqn. 3, element-wise

T_rounded = [round(v, 4) for v in T.tolist()]
y_rounded = [round(v, 4) for v in y.tolist()]
print("worked example:  T =", T_rounded, " y =", y_rounded)
# worked example:  T = [0.8808, 0.1824]  y = [0.8238, -1.5439]

### Step 1 — The highway block and a matched plain block

The highway block has two linear maps: `H` (the transform) and `T` (the gate pre-activation). We initialize the gate's bias *negative* so `T ≈ 0` at the start — the layer carries its input through almost unchanged, behaving like an identity wire until training opens the gate. The plain block is the same width but has no gate and no carry path, the baseline that struggles at depth.

In [ ]:
# --- 1. The highway layer (built by hand). negative gate bias -> carry early (Sec 2.2). ---
class HighwayBlock(nn.Module):
    def __init__(self, n, bias_T=-2.0):
        super().__init__()
        self.H = nn.Linear(n, n)                  # transform
        self.T = nn.Linear(n, n)                  # transform-gate pre-activation
        nn.init.constant_(self.T.bias, bias_T)    # negative bias -> T ~ 0 at start -> carry x

    def forward(self, x):
        h = torch.relu(self.H(x))                 # H(x, W_H)
        t = torch.sigmoid(self.T(x))              # T(x, W_T) in (0,1)
        return h * t + x * (1 - t)                # Eqn. 3:  H*T + x*(1-T)  (element-wise)


# A matched PLAIN block: same width, but no gate and no carry path.
class PlainBlock(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.H = nn.Linear(n, n)

    def forward(self, x):
        return torch.relu(self.H(x))

### Step 2 — Stack 40 blocks into a deep net

We stack `DEPTH = 40` blocks of either kind between a stem and a classification head. The `kind` argument picks highway vs plain, and `bias_T` is exposed so we can later ablate the carry-bias init. This is where depth makes or breaks trainability.

In [ ]:
# --- 2. A deep net of either block type. kind in {"highway","plain"}; bias_T for the ablation. ---
n, K, DEPTH = 32, 3, 40

class DeepNet(nn.Module):
    def __init__(self, kind, bias_T=-2.0):
        super().__init__()
        self.stem = nn.Linear(n, n)
        if kind == "highway":
            self.blocks = nn.Sequential(*[HighwayBlock(n, bias_T) for _ in range(DEPTH)])
        else:
            self.blocks = nn.Sequential(*[PlainBlock(n) for _ in range(DEPTH)])
        self.head = nn.Linear(n, K)

    def forward(self, x):
        stem_out = torch.relu(self.stem(x))
        block_out = self.blocks(stem_out)
        return self.head(block_out)

### Step 3 — Build a tiny dataset and train all three nets

We make a 3-class blob problem (no downloads), then train a plain net, a highway net (`b_T = -2`), and an ablation highway net with `b_T = 0`. The random-guess loss for 3 classes is `ln(3) ≈ 1.0986`: the plain net and the `b_T = 0` ablation both stall there, while the proper highway net descends to near zero — pinning the negative carry-bias init as the reason depth trains.

In [ ]:
# --- 3. A tiny 3-class blob problem (no downloads needed). ---
N = 256
g = torch.Generator().manual_seed(1)
y_lbl = torch.randint(0, K, (N,), generator=g)
centers = torch.randn(K, n, generator=g) * 2.0
X = centers[y_lbl] + torch.randn(N, n, generator=g) * 0.5


def train(kind, bias_T=-2.0, steps=160, lr=0.1):
    torch.manual_seed(0)
    net = DeepNet(kind, bias_T)
    net.train()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lf = nn.CrossEntropyLoss()
    curve = []
    for t in range(steps):
        opt.zero_grad()
        loss = lf(net(X), y_lbl)
        loss.backward()
        opt.step()
        curve.append(loss.item())
    return curve

import math
print("\nrandom-guess loss ln(3) =", round(math.log(K), 4))

plain = train("plain")
highway = train("highway", bias_T=-2.0)
abl0 = train("highway", bias_T=0.0)       # ABLATION: drop the carry-bias init

print("PLAIN   (depth 40, no gate)        final train loss:", round(plain[-1], 4))
print("HIGHWAY (depth 40, b_T=-2)         final train loss:", round(highway[-1], 4))
print("HIGHWAY (depth 40, b_T= 0, ablate) final train loss:", round(abl0[-1], 4))
# PLAIN stalls at ~ln(3)=1.0986 (learns nothing); HIGHWAY descends to ~0.
# The b_T=0 ablation ALSO stalls -> the negative carry-bias init is what makes depth trainable.
# (Exact numbers vary by hardware/torch version; this is our small run, not the paper's result.)

## Visualize the data & results

_At depth 40, does a gated highway MLP train where a matched plain MLP stalls?_

### Step 4 — Rebuild the blocks, net, and dataset for the plot

The visualization re-states the same pieces self-contained so it can run on its own. Same highway/plain blocks, same 40-layer net, and the same 3-class blob data as above — nothing new, just gathered for the loss-curve comparison.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Two matched 40-layer MLPs, identical except for the highway gate + carry-bias init.
# Reproduces the qualitative effect: highway trains where the plain net stalls.
N, K, n, D = 256, 3, 32, 40
g = torch.Generator().manual_seed(1)
y = torch.randint(0, K, (N,), generator=g)
centers = torch.randn(K, n, generator=g) * 2.0
X = centers[y] + torch.randn(N, n, generator=g) * 0.5


class HighwayBlock(nn.Module):
    def __init__(self, n, bias_T):
        super().__init__()
        self.H = nn.Linear(n, n)
        self.T = nn.Linear(n, n)
        nn.init.constant_(self.T.bias, bias_T)     # negative -> carry early

    def forward(self, x):
        h = torch.relu(self.H(x))
        t = torch.sigmoid(self.T(x))
        return h * t + x * (1 - t)                 # Eqn. 3 (element-wise)


class PlainBlock(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.H = nn.Linear(n, n)

    def forward(self, x):
        return torch.relu(self.H(x))               # no gate, no carry


class Net(nn.Module):
    def __init__(self, kind, bias_T=-2.0):
        super().__init__()
        self.stem = nn.Linear(n, n)
        if kind == "highway":
            blk = lambda: HighwayBlock(n, bias_T)
        else:
            blk = lambda: PlainBlock(n)
        self.blocks = nn.Sequential(*[blk() for _ in range(D)])
        self.head = nn.Linear(n, K)

    def forward(self, x):
        stem_out = torch.relu(self.stem(x))
        return self.head(self.blocks(stem_out))

### Step 5 — Train plain vs highway and sample the loss curves

We train both nets for 160 steps and print the loss at 30 evenly spaced checkpoints. The plain net stays pinned near `ln(3)`; the highway net breaks away around step 60 and reaches near zero by step 120 — the qualitative contrast the CODEVIZ panel plots.

In [ ]:
def train(kind, bias_T=-2.0, steps=160, lr=0.1):
    torch.manual_seed(0)
    net = Net(kind, bias_T)
    net.train()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lf = nn.CrossEntropyLoss()
    losses = []
    for t in range(steps):
        opt.zero_grad()
        loss = lf(net(X), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

plain = train("plain")
highway = train("highway", bias_T=-2.0)

idx = np.linspace(0, 159, 30).astype(int)
plain_samples = [[int(i), round(plain[i], 4)] for i in idx]
highway_samples = [[int(i), round(highway[i], 4)] for i in idx]
print("Plain  :", plain_samples)
print("Highway:", highway_samples)
# Plain   -> pinned at ln(3) ~ 1.0887 (never learns at depth 40).
# Highway -> breaks away ~step 60, ~0 by ~step 120.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a 40-layer highway MLP that trains to near-zero loss with the gate
            bias initialized to $-2$. Two changes are proposed: (i) replace every highway block with a plain
            block (relu(H(x)), no gate); (ii) keep the highway gate but set its bias to $0$.
            Predict each curve and say what the comparison demonstrates.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run (i): same depth, plain blocks, no carry path. — _Removes the shortcut entirely — isolates whether the carry path is what enables depth._
- Run (ii): highway blocks but nn.init.constant_(self.T.bias, 0.0). — _Keeps the gate machinery but removes the carry-by-default initialization (§2.2) — isolates the bias trick._
- Compare the three training losses against the random-guess floor $\ln 3 \approx 1.0986$. — _A curve pinned near $\ln 3$ has learned nothing; only a curve that descends well below it is training._

**Answer:** Both ablations stall near the random-guess loss ($\approx 1.09$ for 3 classes) while the
                 negative-bias highway descends to near zero. (i) shows the carry path is necessary; (ii) shows
                 the carry path is not enough on its own — you must initialize the gate to carry
                 ($b_T \lt 0$) so the deep net starts as a chain of near-identity wires. The two together
                 isolate the highway gate and its negative-bias init as the reason deep nets train. Our
                 CODEVIZ panel shows the plain-vs-highway version of this contrast.

</details>

**Problem 2.** A highway unit has input $x = 0.4$, transform $H(x) = 0.9$, and gate pre-activation
            $z = W_T x + b_T = 1.0$. Compute the unit's output $y$ by hand, and say which path dominated.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Sigmoid the gate: $T = \sigma(1.0) = 1/(1 + e^{-1.0}) = 0.7311$. — _The gate is a sigmoid mixing weight in $(0,1)$ — here mostly open._
- Carry gate $C = 1 - T = 0.2689$. — _The carry and transform gates are tied: $C = 1 - T$ (Eqn. 3)._
- Blend: $y = H\cdot T + x\cdot(1-T) = 0.9\times 0.7311 + 0.4\times 0.2689 = 0.6580 + 0.1076 = 0.7656$. — _Eqn. 3, applied to a single unit._

**Answer:** $T = \sigma(1.0) = 0.7311$, so $y = 0.9\times 0.7311 + 0.4\times 0.2689 = 0.7656$. The gate was
                 mostly open ($T \approx 0.73$), so the transform path dominated: the output sits close
                 to $H(x) = 0.9$, pulled down a little by the carried input $0.4$. Had $b_T$ been very negative
                 (gate near closed), $y$ would have stayed near the input $0.4$ instead.

</details>

**Problem 3.** Show algebraically that a highway layer contains both the identity wire and the plain layer as
            special cases, and connect this to why ResNet could later drop the gate.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set $T = 0$ in Eqn. 3: $y = H(x)\cdot 0 + x\cdot(1-0) = x$. — _Gate fully closed — the layer is the identity mapping (Eqn. 4, $T=0$ case)._
- Set $T = 1$: $y = H(x)\cdot 1 + x\cdot(1-1) = H(x)$. — _Gate fully open — the layer is an ordinary plain transform (Eqn. 4, $T=1$ case)._
- Note ResNet fixes the path permanently open as a bare add: $y = x + F(x)$, no gate to learn. — _If the identity path is always useful, a learned gate is extra parameters; hard-wiring it is simpler and still keeps gradients flowing._

**Answer:** At $T = 0$ the layer is the identity $y = x$; at $T = 1$ it is the plain layer
                 $y = H(x)$ (Eqn. 4). A highway layer interpolates smoothly between the two, choosing per unit.
                 ResNet observed that the identity path is the part that matters for trainability and wired it
                 in unconditionally ($y = x + F(x)$), discarding the learned gate — fewer parameters, same
                 gradient highway. Highway is the gated original; ResNet is the un-gated simplification.

</details>